# Prod (a) vs full-frame (d)

| Arm | Mode | LoRA | ref_boost |
| --- | --- | --- | ---: |
| **a** prod | `crop_stitch` | r64 | 3.5 |
| **d** workflow | `full_frame` | full v1.2 | 4.0 |

**No upload required.** §3 loads real demo photos from `data/demo/` (or GitHub).

1. Runtime → **Restart session**
2. Runtime → **Run all**
3. Photos appear in §3; A/B results in §5 (minutes, not seconds)

If §3 errors about “No face detected”, you are still on an old commit — restart and Run all again.


## 1 · Settings


In [ ]:
# === User-facing knobs ===
SEED = 46
STEPS = 8
CFG = 1.0
OUTPUT_LONG_SIDE = 1024
DEBUG = False

# Face selection on multi-person body
BODY_FACE_POLICY = "rightmost"  # night-group demo: swap person on the right
BODY_FACE_INDEX = 0

# Demo mode: NEVER opens an upload dialog.
# Photos are loaded from data/demo/ (or downloaded from GitHub if missing).
USE_DEMO_PAIR = True
DEMO_BODY = "data/demo/body_multi.png"  # real multi-person photo (3 faces)
DEMO_FACE = "data/demo/face.png"         # real identity donor

REPO_BRANCH = "ab/full-frame-author-parity"
PINNED_COMMIT = None                 # None = latest origin/ab/full-frame-author-parity
AUTHOR_AB_ARMS = "a,d"                 # prod vs full-frame d only

print("Settings — prod (a) vs full-frame (d)")
print(f"  USE_DEMO_PAIR={USE_DEMO_PAIR}  (no upload)")
print(f"  DEMO_BODY={DEMO_BODY}")
print(f"  DEMO_FACE={DEMO_FACE}")
print(f"  AUTHOR_AB_ARMS={AUTHOR_AB_ARMS}")
print(f"  BODY_FACE_POLICY={BODY_FACE_POLICY}")
print(f"  REPO_BRANCH={REPO_BRANCH}  PINNED={PINNED_COMMIT or 'HEAD'}")


## 2 · Setup (GPU · Drive · repo · models)


In [ ]:
#@title Setup
from pathlib import Path
import importlib.util
import os
import subprocess

assert Path("/content").exists(), "Open this notebook in Google Colab."

import torch
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Runtime → Change runtime type → GPU (A100 preferred), then Run all.")
print(f"✓ GPU  {torch.cuda.get_device_name(0)}")

from google.colab import drive
print("→ Mounting Drive…")
drive.mount("/content/drive")

REPO_URL = "https://github.com/malihashar/headswap_V2.git"
REPO = Path("/content/headswap_V2")
REPO_BRANCH = globals().get("REPO_BRANCH") or "ab/full-frame-author-parity"
print("→ Syncing repo…")
if not REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=False)
subprocess.run(["git", "-C", str(REPO), "checkout", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"], check=False)
subprocess.run(
    ["git", "-C", str(REPO), "pull", "--ff-only", "origin", REPO_BRANCH],
    check=False,
)
pin = globals().get("PINNED_COMMIT")
if pin:
    print(f"→ Pinning commit {pin}…")
    subprocess.run(["git", "-C", str(REPO), "checkout", str(pin)], check=False)
os.chdir(REPO)
print("✓", subprocess.getoutput(f"git -C {REPO} rev-parse --abbrev-ref HEAD"),
      subprocess.getoutput(f"git -C {REPO} rev-parse --short HEAD"))

!pip install -q -e .

spec = importlib.util.spec_from_file_location("colab_env", REPO / "scripts" / "colab_env.py")
colab_env = importlib.util.module_from_spec(spec)
spec.loader.exec_module(colab_env)
PATHS = colab_env.apply_env(colab_env.default_paths(use_drive=True))

spec_d = importlib.util.spec_from_file_location("colab_demo", REPO / "scripts" / "colab_demo.py")
colab_demo = importlib.util.module_from_spec(spec_d)
spec_d.loader.exec_module(colab_demo)

pin = globals().get("PINNED_COMMIT")
VERSIONS = colab_demo.collect_versions(repo=REPO, comfyui=PATHS["comfyui"], pinned_commit=pin)

print("→ Installing ComfyUI + Krea2 nodes…")
!bash scripts/setup_colab.sh
!bash scripts/setup_krea2_nodes.sh

print("→ Downloading / verifying models (includes full v1.2 LoRA for arm d)…")
_dl = subprocess.run(
    [
        "python", "scripts/download_krea2.py",
        "--comfy", os.environ["COMFYUI_PATH"],
        "--store-dir", os.environ["HEADSWAP_MODEL_STORE"],
        "--staging-dir", os.environ["HEADSWAP_STAGING_DIR"],
        "--backend", "auto",
        "--disable-xet",
        "--include-optional",
    ],
    check=False,
)
if _dl.returncode != 0:
    raise SystemExit(f"Model download failed (exit={_dl.returncode}). Re-run §2.")

colab_demo.verify_models(PATHS["model_store"], check_sizes=True)
VERSIONS = colab_demo.collect_versions(repo=REPO, comfyui=PATHS["comfyui"])
colab_demo.print_versions(VERSIONS)
colab_demo.ok("Setup ready")
print("If FIRST custom-node install on a fresh runtime: Runtime → Restart session, then Run all from §1.")


## 3 · Inputs (demo pair or upload)


In [ ]:
# §3 Inputs — demo photos only (no upload unless you flip USE_DEMO_PAIR=False)
import importlib.util
import shutil
import subprocess
import urllib.request
from pathlib import Path

from IPython.display import display, Markdown
from PIL import Image

REPO = Path("/content/headswap_V2")
assert REPO.is_dir(), "Repo missing — run §2 Setup first."

BRANCH = str(globals().get("REPO_BRANCH") or "ab/full-frame-author-parity")
PIN = globals().get("PINNED_COMMIT")

# Make sure this runtime has the commit that contains data/demo/
print(f"→ Syncing {BRANCH} …")
subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=False)
subprocess.run(
    ["git", "-C", str(REPO), "checkout", "-B", BRANCH, f"origin/{BRANCH}"],
    check=False,
)
subprocess.run(
    ["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH],
    check=False,
)
if PIN:
    subprocess.run(["git", "-C", str(REPO), "checkout", str(PIN)], check=False)
print(
    "✓ repo",
    subprocess.getoutput(f"git -C {REPO} rev-parse --short HEAD"),
    subprocess.getoutput(f"git -C {REPO} rev-parse --abbrev-ref HEAD"),
)

spec = importlib.util.spec_from_file_location("colab_demo", REPO / "scripts" / "colab_demo.py")
colab_demo = importlib.util.module_from_spec(spec)
spec.loader.exec_module(colab_demo)

custom = REPO / "data" / "custom"
custom.mkdir(parents=True, exist_ok=True)
BODY_PATH = custom / "body.png"
FACE_PATH = custom / "face.png"
CACHE = REPO / ".cache" / "headswap_v2"
CACHE.mkdir(parents=True, exist_ok=True)

USE_DEMO_PAIR = bool(globals().get("USE_DEMO_PAIR", True))
DEMO_BODY = str(globals().get("DEMO_BODY", "data/demo/body_multi.png"))
DEMO_FACE = str(globals().get("DEMO_FACE", "data/demo/face.png"))

def _ensure_demo(rel: str, dest: Path) -> Path:
    """Copy from repo, or download from GitHub raw if the file is missing."""
    src = REPO / rel
    if src.is_file() and src.stat().st_size > 1000:
        shutil.copy2(src, dest)
        print(f"✓ copied {rel} → {dest.name} ({dest.stat().st_size} bytes)")
        return dest
    refs = [PIN, BRANCH, "bf56f5f"]
    refs = [str(r) for r in refs if r]
    last_err = None
    dest.parent.mkdir(parents=True, exist_ok=True)
    for ref in refs:
        url = f"https://raw.githubusercontent.com/malihashar/headswap_V2/{ref}/{rel}"
        print(f"→ local missing; downloading {url}")
        try:
            urllib.request.urlretrieve(url, dest)
            if dest.is_file() and dest.stat().st_size > 1000:
                print(f"✓ downloaded → {dest} ({dest.stat().st_size} bytes)")
                return dest
        except Exception as exc:
            last_err = exc
            print(f"  failed: {exc}")
    raise SystemExit(
        f"Could not fetch demo image {rel}. last_error={last_err}\n"
        "Runtime → Restart session, then Run all from §1."
    )


if USE_DEMO_PAIR:
    print("Using demo pair (no upload dialog).")
    _ensure_demo(DEMO_BODY, BODY_PATH)
    _ensure_demo(DEMO_FACE, FACE_PATH)
else:
    from google.colab import files

    print("USE_DEMO_PAIR=False — upload BODY, then FACE.")
    up_body = files.upload()
    if not up_body:
        raise SystemExit("No body uploaded.")
    colab_demo.save_upload(next(iter(up_body.values())), BODY_PATH)
    up_face = files.upload()
    if not up_face:
        raise SystemExit("No face uploaded.")
    colab_demo.save_upload(next(iter(up_face.values())), FACE_PATH)

body_im = Image.open(BODY_PATH).convert("RGB")
face_im = Image.open(FACE_PATH).convert("RGB")
print(f"Loaded body={body_im.size} face={face_im.size}")

try:
    body_face = colab_demo.require_face(body_im, CACHE, "body")
    face_face = colab_demo.require_face(face_im, CACHE, "face")
except colab_demo.DemoError as exc:
    raise SystemExit(
        f"{exc}\n\n"
        "You must use *real* photos with visible faces.\n"
        "If you still see this after a pull: Runtime → Restart session, Run all.\n"
        "Or set USE_DEMO_PAIR=False and upload your own photos."
    ) from exc

display(Markdown("### Inputs ready (demo — no upload needed)"))
print(f"Body: {body_im.size}, faces={body_face['face_count']}")
display(body_im)
print(f"Face: {face_im.size}, faces={face_face['face_count']}")
display(face_im)
colab_demo.ok(f"Saved → {BODY_PATH} , {FACE_PATH}")


## 4 · Run A/B (a vs d)


In [ ]:
import importlib.util
from pathlib import Path

from IPython.display import display, Markdown
from PIL import Image

REPO = Path("/content/headswap_V2")
assert REPO.is_dir(), "Repo missing — run §2 first."

spec_env = importlib.util.spec_from_file_location("colab_env", REPO / "scripts" / "colab_env.py")
colab_env = importlib.util.module_from_spec(spec_env)
spec_env.loader.exec_module(colab_env)
PATHS = colab_env.apply_env(colab_env.default_paths(use_drive=True))
colab_env.ensure_import_path(REPO)

spec = importlib.util.spec_from_file_location("colab_demo", REPO / "scripts" / "colab_demo.py")
colab_demo = importlib.util.module_from_spec(spec)
spec.loader.exec_module(colab_demo)

BODY_PATH = REPO / "data" / "custom" / "body.png"
FACE_PATH = REPO / "data" / "custom" / "face.png"
if not BODY_PATH.is_file() or not FACE_PATH.is_file():
    raise SystemExit(
        f"Missing inputs:\n  {BODY_PATH.exists()=} {BODY_PATH}\n  {FACE_PATH.exists()=} {FACE_PATH}\n"
        "Run §3 first (demo pair or upload)."
    )

REPO_BRANCH = globals().get("REPO_BRANCH") or "ab/full-frame-author-parity"
colab_demo.ensure_repo_branch(REPO, REPO_BRANCH)
# reload after branch sync
spec = importlib.util.spec_from_file_location("colab_demo", REPO / "scripts" / "colab_demo.py")
colab_demo = importlib.util.module_from_spec(spec)
spec.loader.exec_module(colab_demo)

arms = str(globals().get("AUTHOR_AB_ARMS", "a,d")).strip() or "a,d"
policy = str(globals().get("BODY_FACE_POLICY", "largest"))
AB_OUT = REPO / "results" / "_ab_prod_vs_d"

print(f"→ Running prod vs d  arms={arms}  policy={policy}")
print(f"  body={BODY_PATH}  face={FACE_PATH}")
print(f"  out={AB_OUT}")
print("  (first sample loads models — expect several minutes, not seconds)")

AB_RESULT = colab_demo.run_full_frame_author_ab(
    repo=REPO,
    body_path=BODY_PATH,
    face_path=FACE_PATH,
    out_dir=AB_OUT,
    policy=policy,
    arms=arms,
)

RUN_OK = True
RUN_ERROR = None
RUN_DIR = Path(AB_RESULT["out_dir"])
RESULT_PATH = Path(AB_RESULT["side_by_side"]) if AB_RESULT.get("side_by_side") else None
STABLE_PATH = RESULT_PATH
META = {
    "mode": "prod_vs_fullframe_d",
    "wall_s": AB_RESULT.get("wall_s"),
    "report_md": AB_RESULT.get("report_md"),
    "arms": arms,
}
QUALITY = (AB_RESULT.get("report") or {}).get("judgment") or {}

print(f"\n✓ Done in {AB_RESULT.get('wall_s')}s")
print(f"  REPORT → {AB_RESULT.get('report_md')}")
print(f"  side_by_side → {RESULT_PATH}")


## 5 · Results


In [ ]:
from pathlib import Path
from IPython.display import display, Markdown
from PIL import Image
from google.colab import files

REPO = Path("/content/headswap_V2")
out = Path(globals().get("RUN_DIR") or (REPO / "results" / "_ab_prod_vs_d"))

if not globals().get("RUN_OK"):
    raise SystemExit(globals().get("RUN_ERROR") or "§4 did not succeed — scroll up for the error.")

display(Markdown("### Prod (a) vs full-frame (d)"))

# Side-by-side collage if present
sbs = out / "colab_upload" / "side_by_side.png"
if not sbs.is_file():
    # fallback: any case folder
    cands = sorted(out.glob("*/side_by_side.png"))
    sbs = cands[0] if cands else sbs

if sbs.is_file():
    display(Markdown(f"**{sbs.parent.name}** — side by side"))
    display(Image.open(sbs))
else:
    print("No side_by_side.png yet — showing per-arm finals if any.")

# Per-arm finals
shown = 0
for arm_dir in sorted(out.glob("*/*")):
    if not arm_dir.is_dir():
        continue
    for name in ("final_output.png", "result.png"):
        p = arm_dir / name
        if p.is_file():
            display(Markdown(f"**{arm_dir.parent.name} / {arm_dir.name}**"))
            display(Image.open(p))
            shown += 1
            break

if shown == 0 and not sbs.is_file():
    print("No output images found under", out)
    print("Contents:", list(out.rglob("*.png"))[:20])

report = Path((globals().get("AB_RESULT") or {}).get("report_md") or (out / "REPORT.md"))
if report.is_file():
    display(Markdown("### REPORT.md"))
    display(Markdown(report.read_text(encoding="utf-8")[:20000]))
    try:
        files.download(str(report))
    except Exception as exc:
        print("download skipped:", exc)
if sbs.is_file():
    try:
        files.download(str(sbs))
    except Exception as exc:
        print("download skipped:", exc)
